# 02｜清理、EDA 與對帳法（對應課程 M4）

**本 Notebook 的目標**

1. 親眼看到 **JOIN 膨脹**：直接合併一對多的表，金額會被重複計算
2. 一步一步把 9 張表清理成「**一列一筆訂單**」的寬表
3. 把「營收」寫成明確的**指標定義**
4. 做四個面向的 **EDA**：月趨勢、類別、付款方式、縣市
5. 用**對帳法**：同一個數字用兩到三種方法算，必須一致

> **AI 做法 vs 傳統做法**：課堂上你請 AI「整理成一列一筆訂單的寬表」；
> 這裡把同樣的事拆成幾個小函式，每一步都印出處理前後的列數。
> 本 Notebook 的清理邏輯也寫在 `islemart_lib.build_orders_wide()`，之後的 Notebook 直接呼叫它。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import islemart_lib as L

FONT = L.setup_chinese_font()
pd.set_option("display.width", 160)
T = L.load_all()
print("中文字型：", FONT)

# 圖表配色：主色藍、強調橘、輔助灰（整份課程一致）
BLUE, ORANGE, GRAY = "#2E7D9A", "#E07A3F", "#9AA5AD"

## 1. 先犯一次錯：JOIN 膨脹

把 `orders`、`order_items`、`order_payments` 直接合併，再加總售價，看看會發生什麼事。

In [ ]:
naive = (T["orders"]
         .merge(T["order_items"], on="order_id")
         .merge(T["order_payments"], on="order_id"))

correct_total = T["order_items"]["price"].sum()
print(f"orders 列數：{len(T['orders']):,}")
print(f"直接合併後列數：{len(naive):,}")
print(f"直接合併後的售價加總：{naive['price'].sum():,.0f}")
print(f"明細表本身的售價加總：{correct_total:,.0f}")
print(f"多算了：{naive['price'].sum() - correct_total:,.0f}")

**原因**：一筆訂單有 2 列明細、2 列付款，合併後變成 2 × 2 = 4 列，每個商品的售價被算了兩次。

**解法**：明細、付款、評論先**各自彙總成一列一筆訂單**，再和 `orders` 合併。
pandas 的 `merge(..., validate="one_to_one")` 可以在合併時自動檢查，萬一不是一對一就直接報錯。

## 2. 清理步驟一：明細表

1. 標記價格異常（單價 ≥ 同類別中位數 20 倍）
2. 異常的單價以 0 計入「乾淨」的售價，但保留原始售價欄位，方便日後對帳
3. 彙總到訂單層級：品項數、售價加總、運費加總、主要商品與賣家

In [ ]:
items_agg = L.aggregate_items(T["order_items"], T["products"])
print(f"明細 {len(T['order_items']):,} 列 → 彙總後 {len(items_agg):,} 列（一列一筆訂單）")
print("含價格異常的訂單：", items_agg["has_price_outlier"].sum())
items_agg.head(3)

## 3. 清理步驟二：付款表與評論表

- **付款**：加總每筆訂單的付款金額，另外記錄折價券金額與主要付款方式
- **評論**：先刪掉完全重複的列；同一訂單有兩則評論時，保留最後回覆的那一則

In [ ]:
pay_agg = L.aggregate_payments(T["order_payments"])
rev_agg = L.aggregate_reviews(T["order_reviews"])
print(f"付款 {len(T['order_payments']):,} 列 → {len(pay_agg):,} 列")
print(f"評論 {len(T['order_reviews']):,} 列 → {len(rev_agg):,} 列")

## 4. 合併成寬表，並加上衍生欄位

`build_orders_wide()` 依序做：

| 步驟 | 動作 |
| --- | --- |
| 合併 | orders ← customers ← 明細彙總 ← 付款彙總 ← 評論彙總（全部 `validate="one_to_one"`） |
| 統一 | 縣市「臺」→「台」 |
| 衍生 | 下單月份、配送天數、延遲天數、是否延遲 |
| 旗標 | 送達早於出貨、已送達卻無送達日、已取消卻有送達日、付款與明細不符 |

**配送指標只計算「有效送達」的訂單**：狀態是 delivered、有送達日、而且時間先後合理。

In [ ]:
wide = L.build_orders_wide(T)
print(f"寬表：{wide.shape[0]:,} 列 × {wide.shape[1]} 欄")
assert len(wide) == len(T["orders"]), "訂單數改變了！"
wide[["order_id", "order_status", "customer_county", "item_count", "item_total",
      "pay_total", "review_score", "delivery_days", "is_late"]].head()

In [ ]:
# 各種資料品質旗標的筆數
flags = [c for c in wide.columns if c.startswith("flag_")]
wide[flags].sum()

## 5. 指標定義：「營收」到底是什麼？

對照講義 M4 的五個問題，本課範例的選擇是：

| 要決定的事 | 本課選擇 |
| --- | --- |
| 含不含運費？ | 不含，只算商品售價 |
| 哪些訂單要算？ | 只算 `delivered` |
| 從哪張表算？ | 明細表 `price` |
| 折價券怎麼處理？ | 不扣，看標價 |
| 異常高價怎麼辦？ | 排除，並記錄筆數 |

這些選擇寫在 `islemart_lib.revenue()` 裡。**定義改了，所有人的數字一起改**，這就是把指標寫成程式的好處。

In [ ]:
REV = L.revenue(wide)
print(f"營收（本課定義）：{REV:,.0f} 元")

# 換一種定義會差多少？
delivered = wide["order_status"] == "delivered"
alt = {
    "本課定義": REV,
    "不排除價格異常": wide.loc[delivered, "item_total_raw"].sum(),
    "加上運費": REV + wide.loc[delivered, "freight_total"].sum(),
    "用付款表金額": wide.loc[delivered, "pay_total"].sum(),
    "所有狀態都算": wide["item_total"].sum(),
}
pd.Series(alt).map("{:,.0f}".format)

**重點**：同一份資料，「營收」可以差到上千萬元。報告裡的每個數字，都要能說出它用的是哪個定義。

## 6. EDA 一：月趨勢

訂單數與營收是兩個不同單位的量，**分成上下兩張圖**，不要用雙 Y 軸。

In [ ]:
monthly = (wide.groupby("purchase_month")
               .agg(訂單數=("order_id", "count"))
               .join(wide[delivered].groupby("purchase_month")["item_total"].sum().rename("營收")))

fig, axes = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True)
colors = [ORANGE if m.endswith("-11") else BLUE for m in monthly.index]
axes[0].bar(monthly.index, monthly["訂單數"], color=colors, width=0.7)
axes[0].set_title("每月訂單數（依下單月份，含所有狀態）", loc="left")
axes[1].plot(monthly.index, monthly["營收"] / 1e6, color=BLUE, linewidth=2, marker="o", markersize=4)
axes[1].set_title("每月營收（百萬元，已送達訂單）", loc="left")
axes[1].set_ylim(bottom=0)
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)
axes[1].tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()

**你看到什麼？**
- 兩年的 11 月（橘色）都是高峰，2 月是低谷（春節）
- 整體逐月成長
- **2025-12 的營收掉下來**：不是業績變差，而是很多 12 月下旬的訂單還沒送達（M3 的時效性陷阱）

## 7. EDA 二：類別、付款方式、縣市

In [ ]:
cat_rev = (wide[delivered].groupby("main_category")["item_total"].sum()
           .sort_values() / 1e6)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(cat_rev.index, cat_rev.values, color=BLUE, height=0.6)
ax.set_title("各類別營收（百萬元，依訂單主要商品類別）", loc="left")
ax.spines[["top", "right"]].set_visible(False)
for y, v in enumerate(cat_rev.values):
    ax.text(v, y, f" {v:.1f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
pay_mix = (wide["main_payment_type"].value_counts(normalize=True) * 100).round(1)
pay_mix.rename({"credit_card": "信用卡", "mobile_pay": "行動支付", "cod_store_pickup": "超商取貨付款",
                "atm_transfer": "ATM 轉帳", "voucher": "全額折價券"}).to_frame("占比%")

In [ ]:
county = (wide.groupby("customer_county")
              .agg(訂單數=("order_id", "count"))
              .sort_values("訂單數", ascending=False))
county["占比%"] = (county["訂單數"] / county["訂單數"].sum() * 100).round(1)
county.head(10)

## 8. 對帳法：關鍵數字用第二種方法再算一次

挑三個「會進報告」的數字：

1. **總營收**：寬表 vs 直接從明細表算 vs 樞紐分析
2. **不重複顧客數**：寬表 vs 顧客表
3. **最高月份營收**：寬表分組 vs 明細表分組

每一項都用 `assert` 寫成檢查，不一致就會報錯。

In [ ]:
# 方法 B：不經過寬表，直接從原始表依定義計算
it = T["order_items"].copy()
it["outlier"] = L.flag_price_outliers(it, T["products"])
delivered_ids = set(T["orders"].loc[T["orders"]["order_status"] == "delivered", "order_id"])
rev_b = it.loc[it["order_id"].isin(delivered_ids) & ~it["outlier"], "price"].sum()

# 方法 C：樞紐分析（就像在 Excel 裡做）
pivot = pd.pivot_table(wide, values="item_total", index="order_status", aggfunc="sum")
rev_c = pivot.loc["delivered", "item_total"]

# 不重複顧客數
cust_a = wide["customer_unique_id"].nunique()
cust_b = T["customers"]["customer_unique_id"].nunique()

# 最高月份營收
top_a = monthly["營收"].max()
it_m = it.merge(T["orders"][["order_id", "order_purchase_timestamp"]], on="order_id")
it_m = it_m[it_m["order_id"].isin(delivered_ids) & ~it_m["outlier"]]
top_b = it_m.groupby(it_m["order_purchase_timestamp"].dt.strftime("%Y-%m"))["price"].sum().max()

recon = pd.DataFrame([
    ["總營收", REV, rev_b, rev_c],
    ["不重複顧客數", cust_a, cust_b, None],
    ["最高月份營收", top_a, top_b, None],
], columns=["指標", "方法 A：寬表", "方法 B：原始表", "方法 C：樞紐"])
recon["一致"] = recon.apply(lambda r: all(abs(r.iloc[1] - v) < 1 for v in r.iloc[2:4] if pd.notna(v)), axis=1)
recon

In [ ]:
assert recon["一致"].all(), "有數字對不上，請追查原因！"
print("三個關鍵數字對帳通過。")

## 9. 存檔：清理後的寬表

存成 CSV，給 Notebook 03 之後使用，也可以拿去和 AI 的結果對帳。

In [ ]:
wide.to_csv("orders_wide.csv", index=False, encoding="utf-8-sig")
print(f"已存 orders_wide.csv：{len(wide):,} 列")

## 小結

| 課堂上的 AI 做法 | 本 Notebook 的傳統做法 |
| --- | --- |
| 請 AI「先彙總再合併」 | `groupby().agg()` 彙總，`merge(validate="one_to_one")` 自動檢查 |
| 在提示中寫營收定義 | 把定義寫成 `revenue()` 函式 |
| 請 AI 畫 EDA 圖 | matplotlib，每張圖一個重點，不用雙 Y 軸 |
| 用另一個平台對帳 | 同一數字寫三種算法，用 `assert` 自動比對 |

**練習**：把 `PRICE_OUTLIER_FACTOR` 改成 5 或 50，營收會變多少？哪個值比較合理？